In [1]:
import warnings
warnings.filterwarnings(action="ignore")
import logging

from QuantStudio.Core import setDefaultLogLevel
setDefaultLogLevel(level=logging.WARNING)

In [2]:
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

# 计算图框架

QuantStudio 系统以计算图为核心, 系统中主要的计算以有向图的形式表达，图由若干个节点组成，每个节点完成某种定义的计算，节点之间的依赖关系以有向边来表示，边从依赖节点指向被依赖的节点。计算引擎负责调度和执行计算图。

![QuantStudio系统](https://gitee.com/scorpi000/QSImage/raw/master/QuantStudio/QuantStudio系统.jpg)

## Node 节点

每个计算节点必须继承自 `QuantStudio.Core.Node.Node`，对象创建的 `__init__` 方法除了 QuantStudio 对象的三个输入参数外还有一个 `deps` 参数，其为当前节点依赖的节点列表。

Node 定义了以下六个方法，构成了完整的计算生命周期：

| 方法 | 必须实现 | 说明 |
|------|---------|------|
| `init_compute` | 否 | 初始化阶段：遍历时调用，用于注册节点状态和准备任务 |
| `prepare_compute` | 否 | 准备阶段：执行 IO 操作（加载数据、缓存等），不应修改全局状态 |
| `compute` | 否 | **便捷编排入口**：内部调用 `forward_compute` → 递归 deps.compute → `backward_compute`，基础引擎（`Engine`、`ParallelEngine`）直接使用此方法 |
| `forward_compute` | 否 | 前向传播：沿依赖方向（父→子）传递数据，返回局部上下文 |
| `backward_compute` | **是** | 后向传播（**主逻辑**）：沿依赖反方向（子→父）收集结果并执行实际计算 |
| `merge_result` | 否 | 结果合并：并行计算后合并多个分片的结果 |

其中，仅 `backward_compute` 是**必须实现**的（基类抛出 `NotImplementedError`），其余五个方法均有默认实现。最简单的计算节点只需重写 `backward_compute` 即可。

## 计算流程总览

引擎执行计算图时经历三个阶段：

```
┌─────────────────────────────────────────────────────────┐
│ 1. init 阶段 (init_compute)                              │
│    引擎遍历依赖树 → 注册 NodeDict → 填充                  │
│    PrepareNodeDict 和 NodeState                          │
├─────────────────────────────────────────────────────────┤
│ 2. prepare 阶段 (prepare_compute)                        │
│    对 PrepareNodeDict 中的节点执行 IO 操作，支持线程并发   │
├─────────────────────────────────────────────────────────┤
│ 3. compute 阶段                                          │
│    各引擎按自身策略编排 forward/backward：                │
│      - Engine / ParallelEngine: 调用 Node.compute(),     │
│        内部递归完成 forward → deps → backward             │
│      - StackEngine / TreeEngine: 直接编排                │
│        forward_compute 和 backward_compute，             │
│        实现更灵活的遍历和并发控制                          │
└─────────────────────────────────────────────────────────┘
```

`compute` 方法是基础引擎使用的便捷入口，它将 forward/backward 封装为递归调用。而 `StackEngine` 和 `TreeEngine` 直接调用 `forward_compute` / `backward_compute`，以获得对计算流程的更精细控制（如显式栈遍历、节点级并发调度）。

In [3]:
from QuantStudio.Core.Node import Node
# Node 的六个方法
print("=== Node.compute (编排入口) ===")
print(qs_help(Node.compute))
print("-" * 10)
print("=== Node.init_compute ===")
print(qs_help(Node.init_compute))
print("-" * 10)
print("=== Node.prepare_compute ===")
print(qs_help(Node.prepare_compute))
print("-" * 10)
print("=== Node.forward_compute ===")
print(qs_help(Node.forward_compute))
print("-" * 10)
print("=== Node.backward_compute (必须重写) ===")
print(qs_help(Node.backward_compute))
print("-" * 10)
print("=== Node.merge_result ===")
print(qs_help(Node.merge_result))

=== Node.compute (编排入口) ===
类型: function
模块: QuantStudio.Core.Node
签名: Node.compute(self, path: List[str], fwd_data: Any, context: QuantStudio.Core.Node.Context) -> Any
说明文档:
    ⚠️  未找到文档字符串（包括父类）
    该对象可能：
    - 是内置函数/方法（C 实现，无 __doc__）
    - 确实没有文档
----------
=== Node.init_compute ===
类型: function
模块: QuantStudio.Core.Node
签名: Node.init_compute(self, path: List[str], init_data: Any, context: QuantStudio.Core.Node.Context) -> List[Any]
说明文档:
    按照边的方向传递数据执行初始化，可以修改 context 中的全局变量，最好不要有耗时的计算
    
    Args:
        path: 运行至当前节点的路径, 由路径上所有节点 ID 组成的 list
        init_data: 上游传递的数据
        context: 全局上下文对象
    
    Returns:
        产生的向下游传递的数据列表, 如果返回空 list 表示终止继续向下的初始化
----------
=== Node.prepare_compute ===
类型: function
模块: QuantStudio.Core.Node
签名: Node.prepare_compute(self, prepare_data: Any, context: QuantStudio.Core.Node.Context)
说明文档:
    主逻辑计算开始前的准备计算, 不可以修改 context 中的全局变量，最好将 IO 操作在这里实现，只对 context.PrepareNodeDict 中的节点执行该操作
    
    Args:
        prepare_data: 执行准备计算所需的数据, 在 in

## 局部上下文与初始化数据

除了全局上下文 `Context` 外，Node 之间还通过以下类型传递数据：

- **`LocalContext`**：局部运行时上下文。`forward_compute` 返回的第二个值，将作为 `local_context` 参数传递给 `backward_compute`。可携带当前节点特有的运行时信息。
- **`DTLocalContext(LocalContext)`**：时序运算类节点的局部上下文，包含 `DTs: List[datetime]` 字段，表示当前计算的时点序列。
- **`DTInitData`**：时序运算类节点的初始化数据，包含 `DTRange: Tuple[datetime, datetime]` 字段，在 `init_compute` 阶段沿依赖链向下传递时点区间。

这三个类在实际的因子计算和回测中被广泛使用。

In [4]:
from QuantStudio.Core.Node import LocalContext, DTLocalContext, DTInitData

print("=== LocalContext ===")
print(qs_help(LocalContext))
print("-" * 10)
print("=== DTLocalContext ===")
print(qs_help(DTLocalContext))
print("-" * 10)
print("=== DTInitData ===")
print(qs_help(DTInitData))

=== LocalContext ===
类型: class
继承自: __QS_Args__, BaseModel
模块: QuantStudio.Core.Node
构造函数签名: LocalContext.__init__(self, /, **data: 'Any') -> 'None'
构造函数文档:
    Create a new model by parsing and validating input data from keyword arguments.
    
    Raises [`ValidationError`][pydantic_core.ValidationError] if the input data cannot be
    validated to form a valid model.
    
    `self` is explicitly positional-only to allow `self` as a field name.
说明文档:
    节点运算时局部上下文对象
----------
=== DTLocalContext ===
类型: class
继承自: LocalContext, __QS_Args__, BaseModel
模块: QuantStudio.Core.Node
构造函数签名: DTLocalContext.__init__(self, /, **data: 'Any') -> 'None'
构造函数文档:
    Create a new model by parsing and validating input data from keyword arguments.
    
    Raises [`ValidationError`][pydantic_core.ValidationError] if the input data cannot be
    validated to form a valid model.
    
    `self` is explicitly positional-only to allow `self` as a field name.
说明文档:
    时序运算类节点运算时局部上下文对象
----------
=== D

## Context 全局上下文

节点计算方法的入参里有一个全局上下文对象 `Context`，其继承自 `__QS_Args__`，主要用于维护全局信息以及节点的运行时状态。

### 核心字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `Mode` | `Literal["PRD", "DEBUG"]` | 运行模式：生产/调试 |
| `NodeDict` | `Dict[str, Node]` | `{节点QSID: Node}`，引擎在 init 阶段注册的所有节点 |
| `NodeState` | `Dict[str, Any]` | `{节点QSID: Any}`，节点在 `init_compute` 中自行维护的临时状态 |
| `PrepareNodeDict` | `Dict[str, Tuple[str, Any]]` | `{准备ID: (节点QSID, 准备数据)}`，`init_compute` 填充 → `prepare_compute` 消费 |
| `DataCache` | `Optional[Cache]` | 数据缓存对象，在 `prepare_compute` 中用于加载/写入数据 |
| `ExtraData` | `dict` | 其他扩展数据，节点间可自由传递的附加信息 |

### 并行计算相关字段

| 字段 | 类型 | 说明 |
|------|------|------|
| `PID` | `str` | 当前进程 ID，默认为 `"0"` |
| `PIDList` | `List[str]` | 所有运行进程的 ID 列表 |
| `SplitType` | `Literal["连续切分", "间隔切分"]` | 数据切分方式 |
| `Event` | `dict` | `{节点QSID: Event}`，多进程同步事件 |
| `Sub2MainQueue` | `Optional[Any]` | 子进程向主进程发送消息的队列 |
| `TaskExecutor` | `Optional[Executor]` | 节点并行计算使用的线程池执行器 |
| `MaxWorkers` | `int` | 节点并行计算的最大并发量 |

Context 还提供了 `split(n)` 方法（将自身切分为 n 份以支持并行）、`getUpdateData()` / `updateContext()` 方法（子进程完成后同步状态），以及上下文管理器协议（`with Context() as ctx:`）。

In [5]:
from QuantStudio.Core.Node import Context

DemoContext = Context()
display(Markdown(DemoContext.info()))

* Mode(运行模式): typing.Literal['PRD', 'DEBUG'], 默认值 'PRD', 当前取值: 'PRD'
* NodeDict(节点集): typing.Dict[str, QuantStudio.Core.Node.Node], 默认值 {}, {节点ID: Node}, 本次运算的所有 Node, 由计算引擎生成, 当前取值: {}
* NodeState(节点状态): typing.Dict[str, typing.Any], 默认值 {}, {节点ID: Any}, 运算中用于存储节点的临时数据，由节点生成和维护, 当前取值: {}
* PrepareNodeDict(准备节点列表): typing.Dict[str, typing.Tuple[str, typing.Any]], 默认值 {}, {准备ID: (节点ID, Any)}, 需要执行准备操作的节点列表, 当前取值: {}
* PID(当前进程ID): <class 'str'>, 默认值 '0', 当前的运行进程 ID, 默认为 '0', 当前取值: '0'
* PIDList(全部进程ID): typing.List[str], 默认值 ['0'], 所有运行进程 ID 列表, 当前取值: ['0']
* SplitType(切分方式): typing.Literal['连续切分', '间隔切分'], 默认值 '连续切分', 当前取值: '连续切分'
* Event(同步Event): <class 'dict'>, 默认值 {}, {节点ID: Event}, 用于多进程同步的 Event 数据, 当前取值: {}
* Sub2MainQueue: typing.Optional[typing.Any], 默认值 None, 用于子进程向主进程发送消息, 当前取值: None
* TaskExecutor(并行执行器): typing.Optional[concurrent.futures._base.Executor], 默认值 None, 给到节点用于并行计算, 当前取值: None
* MaxWorkers(最大并行数量): <class 'int'>, 默认值 1, 节点执行并行计算的最大并发量, 当前取值: 1
* DataCache(数据缓存): typing.Optional[QuantStudio.Core.Cache.Cache], 默认值 None, 当前取值: None
* ExtraData(其他数据): <class 'dict'>, 默认值 {}, 当前取值: {}

## 计算引擎

计算图的调度和运行由计算引擎对象执行。QuantStudio 提供了多种计算引擎（`Engine`、`StackEngine`、`ParallelEngine`、`TreeEngine`），所有引擎均继承自 `QuantStudio.Core.CalcEngine.Engine`，执行计算的主要方法是 `run`。

```python
Engine.run(node_list, context, init_data_list, fwd_data_list) -> List[Any]
```

`run` 方法按 **init → prepare → compute** 三个阶段顺序执行。关于各引擎的详细说明、参数配置、遍历策略和选择指南，请参见同目录下的 **[计算引擎.ipynb](计算引擎.ipynb)**。

In [6]:
from QuantStudio.Core.CalcEngine import Engine

# Engine.run 是计算图执行的统一入口
print(qs_help(Engine.run))
print("\n详细引擎说明（StackEngine / ParallelEngine / TreeEngine）请参考 计算引擎.ipynb")

类型: function
模块: QuantStudio.Core.CalcEngine
签名: Engine.run(self, node_list: List[QuantStudio.Core.Node.Node], context: QuantStudio.Core.Node.Context, init_data_list: Optional[List[Any]] = None, fwd_data_list: Optional[List[Any]] = None) -> List[Any]
说明文档:
    给定节点列表, 执行所有节点的计算, 返回每个节点的计算结果
    
    Args:
        node_list: 待计算的节点列表
        context: 全局上下文对象
        init_data_list: 初始化数据列表
        fwd_data_list: 前向计算输入数据列表
    
    Returns:
        节点计算的结果列表

详细引擎说明（StackEngine / ParallelEngine / TreeEngine）请参考 计算引擎.ipynb


In [7]:
# 使用计算图框架实现四则运算
from typing import Any, List, Optional

import numpy as np
import pandas as pd

from QuantStudio.Core.Node import Node, Context
from QuantStudio.Core.CalcEngine import Engine

class Num(Node):
    """数字"""
    def __init__(self, value:float, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        if "Name" not in args: args = args | {"Name": str(value)}
        self._Value = value
        return super().__init__(deps=deps, args=args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return self._Value

class Sum(Node):
    """加法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "sum"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.sum(bwd_data_list)

class Sub(Node):
    """减法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "sub"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] - bwd_data_list[1]

class Prod(Node):
    """乘法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "prod"} | args, config_file=config_file, **kwargs)

    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return np.prod(bwd_data_list)

class Div(Node):
    """除法"""
    def __init__(self, deps:List["Node"]=[], args:dict={}, config_file:Optional[str]=None, **kwargs):
        return super().__init__(deps=deps, args={"Name": "div"} | args, config_file=config_file, **kwargs)
    
    def backward_compute(self, path: List[str], bwd_data_list: List[Any], context: Context, local_context:Any=None) -> Any:
        return bwd_data_list[0] / bwd_data_list[1]


Node1 = Prod([Sum([Num(1), Num(2)]), Num(3)], args={"Name": "(1 + 2) * 3"})
Node2 = Sum([Num(3), Prod([Num(3), Num(2)])], args={"Name": "3 + 3 * 2"})

engine = Engine()
NodeList = [Node1, Node2]
Rslt = engine.run(NodeList, Context())
for i, iNode in enumerate(NodeList):
    print(iNode.Name, "=", Rslt[i])

(1 + 2) * 3 = 9
3 + 3 * 2 = 9


In [ ]:
# 如需可视化计算图，请先安装 mermaid-python 包
# %pip install mermaid-python

## 计算图可视化

QuantStudio 提供了两个工具函数将计算图转换为 Mermaid 流程图：

- **`node2dict(node_list)`**：递归遍历 `Node` 列表，将其转换为嵌套字典。Key 格式为 `"QSID:Name"`，重复出现的节点（共享依赖）对应的值为 `None`。
- **`dict2mermaid(nested_dict, direction)`**：将嵌套字典转换为 Mermaid 流程图代码。`direction` 控制图的绘制方向，可选 `"TD"`（上→下）、`"LR"`（左→右）、`"BT"`（下→上）、`"RL"`（右→左），默认为 `"TD"`。

In [8]:
# 计算图的可视化
from mermaid import Mermaid
from QuantStudio.Tools.Visualization import node2dict, dict2mermaid

# 将 Node 列表转换为嵌套字典
NodeDict = node2dict([Node1, Node2])
# 将嵌套字典转换为 Mermaid 流程图代码 (TD: 上→下, LR: 左→右)
NodeMermaid = dict2mermaid(NodeDict, direction="TD")
display(Mermaid(NodeMermaid))